In [0]:
df = spark.read.format("csv").option("inferSchema",True).option("header", "true").load("/Volumes/databricks-pyspark/databricks-pyspark-schema/internal-v1/superstore (1).csv")        

In [0]:
df.createOrReplaceTempView("sampleview")

In [0]:
%sql
select count(distinct(ID)) from sampleview;

In [0]:
dbutils.widgets.dropdown("time_period","weekly",["weekly","monthly"])

In [0]:
%sql
select distinct(Order_Date) from sampleview;

In [0]:
from datetime import datetime, timedelta

# Get the selected time period
time_period = dbutils.widgets.get("time_period")

# Get the max date in the dataset (exclude invalid date formats)
max_date = spark.sql("select max(Order_Date) as max_date from sampleview where Order_Date not like '%/%'").collect()[0]['max_date']
max_date = datetime.strptime(str(max_date), "%Y-%m-%d")

if time_period == "weekly":
    # Last week: Monday to Sunday
    last_sunday = max_date - timedelta(days=max_date.weekday() + 1)
    last_monday = last_sunday - timedelta(days=6)
    start_date = last_monday.strftime("%Y-%m-%d")
    end_date = last_sunday.strftime("%Y-%m-%d")
elif time_period == "monthly":
    # Last month: first to last day
    first_of_this_month = max_date.replace(day=1)
    last_month_end = first_of_this_month - timedelta(days=1)
    last_month_start = last_month_end.replace(day=1)
    start_date = last_month_start.strftime("%Y-%m-%d")
    end_date = last_month_end.strftime("%Y-%m-%d")

display({"start_date": start_date, "end_date": end_date})

In [0]:
from pyspark.sql.functions import col, expr

# clean column names first
for c in df.columns:
    df = df.withColumnRenamed(c, c.strip().replace(" ", "_"))

# remove header rows if repeated inside data
df = df.filter(col("Sales") != "Sales")

# convert safely
new_df = df \
    .withColumn("Order_Date", expr("try_to_date(Order_Date, 'dd/MM/yyyy')")) \
    .withColumn("Ship_Date", expr("try_to_date(Ship_Date, 'dd/MM/yyyy')")) \
    .withColumn("Sales", expr("try_cast(Sales as double)")) \
    .withColumn("Quantity", expr("try_cast(Quantity as int)")) \
    .withColumn("Discount", expr("try_cast(Discount as double)")) \
    .withColumn("Profit", expr("try_cast(Profit as double)"))

new_df.printSchema()
display(new_df)